## testing some of the access model's analysis using references

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [ ]:
from scipy import stats

In [ ]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [ ]:
### Functions needed for the analysis

In [ ]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    # ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False, cbar_orientation='vertical', hatch_type = 'insig'):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            if hatch_type == 'insig':
                pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            elif hatch_type == 'sig':
                pval_plot = np.ma.masked_greater(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = cbar_orientation, shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [ ]:
from functions import preproc_funcs as funcs

In [ ]:
from functions import xr_lowess

In [ ]:
#### import data

In [ ]:
ts_trans = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp5_ts.nc').ts
pr_trans = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp5_pr.nc').pr*86400*30

In [ ]:
ts_trans1 = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp1_ts.nc').ts
pr_trans1 = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp1_pr.nc').pr*86400*30

In [ ]:
ts_trans2 = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp2_ts.nc').ts
pr_trans2 = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp2_pr.nc').pr*86400*30

In [ ]:
ts_trans5o = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp5o_ts.nc').ts
pr_trans5o = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp5o_pr.nc').pr*86400*30

In [ ]:
ts_stable = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable.nc').ts
pr_stable = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable_pr.nc').pr*86400*30

In [ ]:
pr_trans_noanom = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/ACCESS-ESM1-5_ssp5_pr_original.nc').pr*86400*30

In [ ]:
pr_stable_noanom = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable_pr_original.nc').resample(time = 'AS-JUN').mean('time').load().pr*86400*30

### analysis

In [ ]:
weights_model = np.cos(np.deg2rad(ts_stable.lat))
weights_model.name='weights'
weights_model

In [ ]:
gmst_trans1 = ts_trans1.weighted(weights_model).mean(('lat', 'lon'))
# gmst_trans2 = ts_trans2.weighted(weights_model).mean(('lat', 'lon'))
gmst_trans5o = ts_trans5o.weighted(weights_model).mean(('lat', 'lon'))
gmst_trans = ts_trans.weighted(weights_model).mean(('lat', 'lon'))

In [ ]:
gmst_stable = ts_stable.weighted(weights_model).mean(('lat', 'lon'))

In [ ]:
plt.plot(gmst_trans1.time.dt.year, gmst_trans1.quantile(0.5, 'model'), color='tab:blue', label='ssp126')
plt.fill_between(gmst_trans1.time.dt.year, gmst_trans1.quantile(0.1, 'model'), gmst_trans1.quantile(0.9, 'model'), color='tab:blue', alpha=0.3)

plt.plot(gmst_trans5o.time.dt.year, gmst_trans5o.quantile(0.5, 'model'), color='tab:purple', label='ssp534-over')
plt.fill_between(gmst_trans5o.time.dt.year, gmst_trans5o.quantile(0.1, 'model'), gmst_trans5o.quantile(0.9, 'model'), color='tab:purple', alpha=0.3)

plt.plot(gmst_trans.time.dt.year, gmst_trans.quantile(0.5, 'model'), color='k', label='ssp585')
plt.fill_between(gmst_trans.time.dt.year, gmst_trans.quantile(0.1, 'model'), gmst_trans.quantile(0.9, 'model'), color='k', alpha=0.3)

plt.legend(frameon=False, fontsize=10)
plt.ylabel('Temperature anomaly (deg.C)')
plt.xlabel('Year')
plt.minorticks_on()
plt.gca().tick_params(bottom=False, which='minor', axis='x')

In [ ]:
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2030'), label='B2030', color='gold')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2035'), label='B2035', color='tab:orange')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2040'), label='B2040', color='red')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2045'), label='B2045', color='tab:red')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2050'), label='B2050', color='darkred')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2055'), label='B2055', color='maroon')
plt.plot(gmst_stable.time.dt.year, gmst_stable.sel(model = 'B2060'), label='B2060', color='tab:brown')

plt.plot(gmst_trans.time.dt.year, gmst_trans.quantile(0.5, 'model'), color='k', label='ssp585')
plt.fill_between(gmst_trans.time.dt.year, gmst_trans.quantile(0.1, 'model'), gmst_trans.quantile(0.9, 'model'), color='k', alpha=0.3)

plt.legend(frameon=False, fontsize=10)
plt.ylabel('Temperature anomaly (deg.C)')
plt.xlabel('Year')
plt.minorticks_on()
plt.gca().tick_params(bottom=False, which='minor', axis='x')

In [ ]:
def rolling_window_smoothing(da, window_size=30):
    pad_size=window_size//2
    padded_data = da.pad(time=(pad_size, pad_size), mode='edge')
    smoothed_data = padded_data.rolling(time=window_size, center=True).mean('time').isel(time = slice(int(window_size/2),-int(window_size/2)))
    return smoothed_data

In [ ]:
def rolling_window_std(da, window_size=30):
    pad_size=window_size//2
    padded_data = da.pad(time=(pad_size, pad_size), mode='edge')
    std_data = padded_data.rolling(time=window_size, center=True).std('time').isel(time = slice(int(window_size/2),-int(window_size/2)))
    return std_data

In [ ]:
base = ts_trans.isel(time = 0, model=0)
base

In [ ]:
def additional_sampling_stabilisation(da):
    i = 0
    da_list = []
    while i <= 960:
        sample_da = da.isel(time = slice(i,i+40)).mean('time')
        da_list.append(sample_da)
        i += 40
    print(len(da_list))
    return xr.concat(da_list, dim=np.arange(0, 25, 1)).drop('model').rename(dict(concat_dim='model'))


In [ ]:
additional_sampling_stabilisation(ts_stable.sel(model = 'B2060').sel(time = slice('2060', '3060')))

In [ ]:
plot_list = [
    # ts_trans.where((gmst_trans < 3.1) & (gmst_trans > 2.9)).mean(('time', 'model')),
    # ts_stable.where((gmst_stable < 3.1) & (gmst_stable > 2.9)).mean(('time', 'model')),
    # ts_trans.where((gmst_trans < 2.1) & (gmst_trans > 1.9)).mean(('time', 'model')),
    # ts_stable.where((gmst_stable < 2.1) & (gmst_stable > 1.9)).mean(('time', 'model')),
    # ts_trans.where((gmst_trans < 1.7) & (gmst_trans > 1.3)).mean(('time', 'model')),
    # ts_stable.where((gmst_stable < 1.7) & (gmst_stable > 1.3)).mean(('time', 'model')),
    ts_trans.sel(time = slice('2045', '2075')).mean(('time', 'model')),
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2060').sel(time = slice('2060', '3060'))).mean('model'),
    ts_trans.sel(time = slice('2030', '2060')).mean(('time', 'model')),
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2045').sel(time = slice('2045', '3045'))).mean('model'),
    ts_trans.sel(time = slice('2015', '2045')).mean(('time', 'model')),
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean('model'),
]

In [ ]:
titles_list = [
    # 'ssp585 GWL: 3degC',
    # 'NZ GWL: 3degC',
    # 'ssp585 GWL: 2degC',
    # 'NZ GWL: 2degC',
    # 'ssp585 GWL: 1.5degC',
    # 'NZ GWL: 1.5degC',
    'ssp585 [2045-2075]',
    'NZ B2060',
    'ssp585 [2030-2060]',
    'NZ B2045',
    'ssp585 [2015-2045]',
    'NZ B2030',
]

In [ ]:
xx, yy = np.meshgrid(base.lon, base.lat)

In [ ]:
plot_maps(xx, yy, plot_list, titles_list, labels=['a', 'b', 'd', 'e', 'g', 'h'], cmap='seismic', levels=np.arange(-4, 4.2, 0.2), cbar_label = 'Temperature anomaly (degC)', pval = [], nrows=3, ncols=2, figsize=(8,8), land_mask_list = [], add_patch=True, cbar_orientation='vertical')

In [ ]:
ax = plt.axes(projection=ccrs.Miller(central_longitude=180))
test1 = ts_trans.sel(time = slice('2015', '2045')).mean(('model', 'time')).sel(lat = slice(-80, 80), lon = slice(90, -60+360))
(test1 - test1.mean(('lat', 'lon'))).plot(vmin=-1, cmap='RdBu_r', transform=ccrs.PlateCarree())
plot_background(ax)
ax.add_feature(cfeature.LAND, color='k', zorder=1)

In [ ]:
ax = plt.axes(projection=ccrs.Miller(central_longitude=180))
test2 = additional_sampling_stabilisation(ts_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean('model').sel(lat = slice(-80, 80), lon = slice(90, -60+360))
(test2 - test2.mean(('lat', 'lon'))).plot(vmin=-1, cmap='RdBu_r', transform=ccrs.PlateCarree())
plot_background(ax)
ax.add_feature(cfeature.LAND, color='k', zorder=1)

In [ ]:
ax = plt.axes(projection=ccrs.Miller(central_longitude=180))
((test2 - test1) - (test2 - test1).mean(('lat', 'lon'))).plot(vmin=-0.5, cmap='RdBu_r', transform=ccrs.PlateCarree())
# (test2 - test1).plot(vmin=-0.5, cmap='RdBu_r', transform=ccrs.PlateCarree())
plot_background(ax)
ax.add_feature(cfeature.LAND, color='k', zorder=1)

In [ ]:
# making 3d version of mann whitney u test
def ks_1d(dist1, dist2):
    return float(stats.ks_2samp(dist1, dist2).pvalue)


def ks_3d(da1, da2, dim):
    return xr.apply_ufunc(ks_1d, da1, da2, input_core_dims=[[dim], [dim]], exclude_dims={dim, dim}, vectorize=True, dask='parallelized')

In [ ]:
plot_list = [
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2060').sel(time = slice('2060', '3060'))).mean('model') - ts_trans.sel(time = slice('2045', '2075')).mean(('time', 'model')),
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2045').sel(time = slice('2045', '3045'))).mean('model') - ts_trans.sel(time = slice('2030', '2060')).mean(('time', 'model')),
    additional_sampling_stabilisation(ts_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean('model') - ts_trans.sel(time = slice('2015', '2045')).mean(('time', 'model')),
]

In [ ]:
pval_list = [
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2060').sel(time = slice('2060', '3060'))), ts_trans.sel(time = slice('2045', '2075')).mean(('time')), dim='model'),
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2045').sel(time = slice('2045', '3045'))), ts_trans.sel(time = slice('2030', '2060')).mean(('time')), dim='model'),
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))), ts_trans.sel(time = slice('2015', '2045')).mean(('time')), dim='model'),
]

In [ ]:
titles_list = [
    'DIFF GWL: 2060',
    'DIFF GWL: 2045',
    'DIFF GWL: 2030',
]


In [ ]:
plot_maps(xx, yy, plot_list, titles_list, labels=['c', 'f', 'i'], cmap='RdBu_r', levels=np.arange(-1, 1.1, 0.1), cbar_label = 'Temperature anomaly (degC)', pval = pval_list, nrows=3, ncols=1, figsize=(4.5,8), land_mask_list = [], add_patch=True, cbar_orientation='vertical')

In [ ]:
pr_trans['model'] = gmst_trans['model']
pr_stable['model'] = gmst_stable['model']

In [ ]:
pr_standardizer = pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model') 

In [ ]:
plot_list = [
    # pr_trans.where((gmst_trans < 3.1) & (gmst_trans > 2.9)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    # pr_stable.where((gmst_stable < 3.1) & (gmst_stable > 2.9)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    # pr_trans.where((gmst_trans < 2.1) & (gmst_trans > 1.9)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    # pr_stable.where((gmst_stable < 2.1) & (gmst_stable > 1.9)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    # pr_trans.where((gmst_trans < 1.7) & (gmst_trans > 1.3)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    # pr_stable.where((gmst_stable < 1.7) & (gmst_stable > 1.3)).mean(('time', 'model'))/pr_trans.sel(time = slice('1850', '1900')).std('time').mean('model'),
    pr_trans.sel(time = slice('2045', '2075')).mean(('time', 'model'))/pr_standardizer,
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2060').sel(time = slice('2060', '3060'))).mean('model')/pr_standardizer,
    pr_trans.sel(time = slice('2030', '2060')).mean(('time', 'model'))/pr_standardizer,
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2045').sel(time = slice('2045', '3045'))).mean('model')/pr_standardizer,
    pr_trans.sel(time = slice('2015', '2045')).mean(('time', 'model'))/pr_standardizer,
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean('model')/pr_standardizer,
]

In [ ]:
titles_list = [
    # 'ssp585 GWL: 3degC',
    # 'NZ GWL: 3degC',
    # 'ssp585 GWL: 2degC',
    # 'NZ GWL: 2degC',
    # 'ssp585 GWL: 1.5degC',
    # 'NZ GWL: 1.5degC',
    'ssp585 [2045-2075]',
    'NZ B2060',
    'ssp585 [2030-2060]',
    'NZ B2045',
    'ssp585 [2015-2045]',
    'NZ B2030',
]

In [ ]:
xx, yy = np.meshgrid(base.lon, base.lat)

In [ ]:
plot_maps(xx, yy, plot_list, titles_list, labels=['a', 'b', 'd', 'e', 'g', 'h'], cmap='BrBG', levels=np.arange(-3, 3.2, 0.2), cbar_label = 'Rainfall anomaly (mm/month)', pval = [], nrows=3, ncols=2, figsize=(8,8), land_mask_list = [], add_patch=True, cbar_orientation='vertical')

In [ ]:
plot_list = [
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2060').sel(time = slice('2060', '3060'))).mean('model')/pr_standardizer - pr_trans.sel(time = slice('2045', '2075')).mean(('time', 'model'))/pr_standardizer,
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2045').sel(time = slice('2045', '3045'))).mean('model')/pr_standardizer - pr_trans.sel(time = slice('2030', '2060')).mean(('time', 'model'))/pr_standardizer,
    additional_sampling_stabilisation(pr_stable.sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean('model')/pr_standardizer - pr_trans.sel(time = slice('2015', '2045')).mean(('time', 'model'))/pr_standardizer,
]

In [ ]:
pval_list = [
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2060').sel(time = slice('2060', '3060')))/pr_standardizer, ts_trans.sel(time = slice('2045', '2075')).mean(('time'))/pr_standardizer, dim='model'),
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2045').sel(time = slice('2045', '3045')))/pr_standardizer, ts_trans.sel(time = slice('2030', '2060')).mean(('time'))/pr_standardizer, dim='model'),
    funcs.mannwhitneyu3d(additional_sampling_stabilisation(ts_stable.sel(model = 'B2030').sel(time = slice('2030', '3030')))/pr_standardizer, ts_trans.sel(time = slice('2015', '2045')).mean(('time'))/pr_standardizer, dim='model'),
]

In [ ]:
titles_list = [
    'DIFF GWL: 2060',
    'DIFF GWL: 2045',
    'DIFF GWL: 2030',
]


In [ ]:
plot_maps(xx, yy, plot_list, titles_list, labels=['c', 'f', 'i'], cmap='BrBG', levels=np.arange(-1.0, 1.1, 0.1), cbar_label = 'Rainfall anomaly (mm/month)', pval = pval_list, nrows=3, ncols=1, figsize=(4.5,8), land_mask_list = [], add_patch=True, cbar_orientation='vertical', hatch_type='insig')

In [ ]:
ax = plt.axes(projection = ccrs.Robinson(central_longitude=180))
pr_trans_noanom.sel(lat=slice(-31, 31), lon = slice(110, -60+360)).sel(time = slice('1850', '1900')).mean(('time', 'model')).plot(vmax = 300, cmap='YlGnBu', transform=ccrs.PlateCarree(), cbar_kwargs = dict(label='Rainfall (mm/month)', shrink=0.7))
plot_background(ax)
plt.title('1850 - 1900')
plt.text(0.1, 1.05, 'a', size=16, fontweight='bold', transform=plt.gca().transAxes)
plt.gca().add_patch(mpatches.Rectangle(xy=[130, 0], width=150, height=20,
                                facecolor='none', edgecolor='k', lw=2.0,
                                transform=ccrs.PlateCarree()))
plt.gca().add_patch(mpatches.Rectangle(xy=[130, -20], width=90, height=20,
                                facecolor='none', edgecolor='k', lw=2.0,
                                transform=ccrs.PlateCarree()))

In [ ]:
pr_trans_noanom['model'] = gmst_trans['model']

In [ ]:
ax = plt.axes(projection = ccrs.Robinson(central_longitude=180))
# pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).where((gmst_trans > 2.9) & (gmst_trans < 3.1)).mean(('time', 'model')).plot(vmax = 300, cmap='YlGnBu', transform=ccrs.PlateCarree(), cbar_kwargs = dict(label='Rainfall (mm/month)', shrink=0.7))
# pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).where((gmst_trans > 2.9) & (gmst_trans < 3.1)).mean(('time', 'model')).plot.contour(levels=[250], colors='tab:red', transform=ccrs.PlateCarree())
# pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('1850', '1900')).mean(('time', 'model')).plot.contour(levels=[250], colors='gold', transform=ccrs.PlateCarree())
pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('2015', '2045')).mean(('time', 'model')).plot(vmax = 300, cmap='YlGnBu', transform=ccrs.PlateCarree(), cbar_kwargs = dict(label='Rainfall (mm/month)', shrink=0.7))
pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('2015', '2045')).mean(('time', 'model')).plot.contour(levels=[250], colors='tab:red', transform=ccrs.PlateCarree())
pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('1850', '1900')).mean(('time', 'model')).plot.contour(levels=[250], colors='gold', transform=ccrs.PlateCarree())
plt.title('ssp585 [2015-2045]')
plt.text(0.1, 1.05, 'f', size=16, fontweight='bold', transform=plt.gca().transAxes)
plot_background(ax)
# plt.gca().add_patch(mpatches.Rectangle(xy=[130, 0], width=150, height=20,
#                                 facecolor='none', edgecolor='k', lw=2.0,
#                                 transform=ccrs.PlateCarree()))
# plt.gca().add_patch(mpatches.Rectangle(xy=[130, -20], width=90, height=20,
#                                 facecolor='none', edgecolor='k', lw=2.0,
#                                 transform=ccrs.PlateCarree()))

In [ ]:
ax = plt.axes(projection = ccrs.Robinson(central_longitude=180))
# pr_stable_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).where((gmst_stable > 2.9) & (gmst_stable < 3.1)).mean(('time', 'model')).plot(vmax = 300, cmap='YlGnBu', transform=ccrs.PlateCarree(), cbar_kwargs = dict(label='Rainfall (mm/month)', shrink=0.7))
# pr_stable_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).where((gmst_stable > 2.9) & (gmst_stable < 3.1)).mean(('time', 'model')).plot.contour(levels=[250], colors='tab:red', transform=ccrs.PlateCarree())
# pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('1850', '1900')).mean(('time', 'model')).plot.contour(levels=[250], colors='gold', transform=ccrs.PlateCarree())
additional_sampling_stabilisation(pr_stable_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean(('model')).plot(vmax = 300, cmap='YlGnBu', transform=ccrs.PlateCarree(), cbar_kwargs = dict(label='Rainfall (mm/month)', shrink=0.7))
additional_sampling_stabilisation(pr_stable_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(model = 'B2030').sel(time = slice('2030', '3030'))).mean(('model')).plot.contour(levels=[250], colors='tab:red', transform=ccrs.PlateCarree())
pr_trans_noanom.sel(lat=slice(-30, 30), lon = slice(110, -60+360)).sel(time = slice('1850', '1900')).mean(('time', 'model')).plot.contour(levels=[250], colors='gold', transform=ccrs.PlateCarree())
plt.title('NZ B2030')
plt.text(0.1, 1.05, 'g', size=16, fontweight='bold', transform=plt.gca().transAxes)
plot_background(ax)

In [ ]:
test = pr_trans_noanom.sel(time = slice('1850', '1900')).mean(('time')).sel(lat = slice(0,20), lon = slice(130, -80+360))
test

In [ ]:
# def find_itcz_spcz_position_model_spread(pr_da, type='trans'):
#     if type == 'trans':
#         input_da = pr_da.mean('time')
#         itcz_box = input_da.sel(lat = slice(0,20), lon = slice(130, -80+360))
#         spcz_box = input_da.sel(lat = slice(-20,0), lon = slice(130, -140+360))
#         itcz_loc = itcz_box.lat.where(itcz_box == itcz_box.max('lat')).mean(('lat', 'lon'))
#         spcz_loc = spcz_box.lat.where(spcz_box == spcz_box.max('lat')).mean(('lat', 'lon'))
#     elif type == 'stable':
#         i = 0
#         temporary_itcz_loc = []
#         temporary_spcz_loc = []
#         while i <= 900:
#             input_da = pr_da.isel(time = slice(i, i+100)).mean('time')
#             itcz_box = input_da.sel(lat = slice(0,20), lon = slice(130, -80+360))
#             spcz_box = input_da.sel(lat = slice(-20,0), lon = slice(130, -140+360))
#             temporary_itcz_loc.append(itcz_box.lat.where(itcz_box == itcz_box.max('lat')).mean(('lat', 'lon')).to_numpy())
#             temporary_spcz_loc.append(spcz_box.lat.where(spcz_box == spcz_box.max('lat')).mean(('lat', 'lon')).to_numpy())
#             i += 100
#         itcz_loc = np.concatenate(temporary_itcz_loc, axis=None)
#         spcz_loc = np.concatenate(temporary_spcz_loc, axis=None)
#     return itcz_loc[~np.isnan(itcz_loc)], spcz_loc[~np.isnan(spcz_loc)]




def find_itcz_spcz_position_model_spread(pr_da, type='trans'):
    if type == 'trans':
        input_da = pr_da.mean('time')
    elif type == 'stable':
        input_da = pr_da
    itcz_box = input_da.sel(lat = slice(0,20), lon = slice(130, -80+360))
    spcz_box = input_da.sel(lat = slice(-20,0), lon = slice(130, -140+360))
    itcz_loc = itcz_box.lat.where(itcz_box == itcz_box.max('lat')).mean(('lat', 'lon'))
    spcz_loc = spcz_box.lat.where(spcz_box == spcz_box.max('lat')).mean(('lat', 'lon'))
    return itcz_loc[~np.isnan(itcz_loc)], spcz_loc[~np.isnan(spcz_loc)]

In [ ]:
def find_nino34_magnitude(ts_da, type='trans'):
    if type == 'trans':
        input_da = pr_da.mean('time')
    elif type == 'stable':
        input_da = pr_da
    nino34_box = input_da.sel(lat = slice(0,20), lon = slice(130, -80+360))
    spcz_box = input_da.sel(lat = slice(-20,0), lon = slice(130, -140+360))
    itcz_loc = itcz_box.lat.where(itcz_box == itcz_box.max('lat')).mean(('lat', 'lon'))
    spcz_loc = spcz_box.lat.where(spcz_box == spcz_box.max('lat')).mean(('lat', 'lon'))
    return itcz_loc[~np.isnan(itcz_loc)], spcz_loc[~np.isnan(spcz_loc)]

In [ ]:
itcz_loc_pi, spcz_loc_pi = find_itcz_spcz_position_model_spread(pr_trans_noanom.sel(time = slice('1850', '1900')))
itcz_loc_15, spcz_loc_15 = find_itcz_spcz_position_model_spread(pr_trans_noanom.sel(time = slice('2015', '2045')))
itcz_loc_20, spcz_loc_20 = find_itcz_spcz_position_model_spread(pr_trans_noanom.sel(time = slice('2030', '2060')))
itcz_loc_30, spcz_loc_30 = find_itcz_spcz_position_model_spread(pr_trans_noanom.sel(time = slice('2045', '2075')))
itcz_loc_post21, spcz_loc_post21 = find_itcz_spcz_position_model_spread(pr_trans_noanom.sel(time = slice('2250', '2300')))

In [ ]:
itcz_loc_15_stable, spcz_loc_15_stable = find_itcz_spcz_position_model_spread(additional_sampling_stabilisation(pr_stable_noanom.sel(model = 'B2030').sel(time = slice('2030', '3030'))), type='stable')
itcz_loc_20_stable, spcz_loc_20_stable = find_itcz_spcz_position_model_spread(additional_sampling_stabilisation(pr_stable_noanom.sel(model = 'B2045').sel(time = slice('2045', '3045'))), type='stable')
itcz_loc_30_stable, spcz_loc_30_stable = find_itcz_spcz_position_model_spread(additional_sampling_stabilisation(pr_stable_noanom.sel(model = 'B2060').sel(time = slice('2060', '3060'))), type='stable')

In [ ]:
fig = plt.figure(figsize=(6,6))
bp1 = plt.boxplot([itcz_loc_pi, itcz_loc_15, itcz_loc_20, itcz_loc_30, itcz_loc_post21], whis=[5,95], widths=0.3, boxprops=dict(color='k', lw=2.0), medianprops=dict(color='k'), meanprops=dict(marker='^'), showmeans=True, flierprops=dict(marker='+'), positions=[0, 2, 5, 8, 11])
bp2 = plt.boxplot([itcz_loc_15_stable, itcz_loc_20_stable, itcz_loc_30_stable], whis=[5,95], widths=0.3, boxprops=dict(color='tab:blue', lw=2.0), medianprops=dict(color='tab:blue'), meanprops=dict(marker='^'), showmeans=True, whiskerprops=dict(color='tab:blue'), capprops=dict(color='tab:blue'), flierprops=dict(marker='+', color='tab:blue'), positions=[3, 6, 9])
plt.gca().set_xticks([0, 2.5, 5.5, 8.5, 11.5])
plt.gca().set_xticklabels(['1850-1900', 'B2030', 'B2045', 'B2060', '2250-2300'])
plt.ylabel('Latitude ($^{o}$N)')
# plt.xlim(-1, 11)
plt.axvline(1, color='k', lw=0.5)
plt.axvline(4, color='k', lw=0.5)
plt.axvline(7, color='k', lw=0.5)
plt.axvline(10, color='k', lw=0.5)
plt.axhline(itcz_loc_pi.mean(), color='tab:green', lw=0.5)
plt.title('ITCZ location')
plt.minorticks_on()
plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.xlim(-2, 13)
plt.legend([bp1["boxes"][0], bp2["boxes"][0]], ['Transient', 'NZ'], loc='lower left', frameon=False, fontsize=10)


In [ ]:
fig = plt.figure(figsize=(6,6))
bp1 = plt.boxplot([spcz_loc_pi, spcz_loc_15, spcz_loc_20, spcz_loc_30, spcz_loc_post21], whis=[5,95], widths=0.3, boxprops=dict(color='k', lw=2.0), medianprops=dict(color='k'), meanprops=dict(marker='^'), showmeans=True, flierprops=dict(marker='+'), positions=[0, 2, 5, 8, 11])
bp2 = plt.boxplot([spcz_loc_15_stable, spcz_loc_20_stable, spcz_loc_30_stable], whis=[5,95], widths=0.3, boxprops=dict(color='tab:blue', lw=2.0), medianprops=dict(color='tab:blue'), meanprops=dict(marker='^'), showmeans=True, whiskerprops=dict(color='tab:blue'), capprops=dict(color='tab:blue'), flierprops=dict(marker='+', color='tab:blue'), positions=[3, 6, 9])
plt.gca().set_xticks([0, 2.5, 5.5, 8.5, 11.5])
plt.gca().set_xticklabels(['1850-1900', 'B2030', 'B2045', 'B2060', '2250-2300'])
plt.ylabel('Latitude ($^{o}$N)')
plt.axvline(1, color='k', lw=0.5)
plt.axvline(4, color='k', lw=0.5)
plt.axvline(7, color='k', lw=0.5)
plt.axvline(10, color='k', lw=0.5)
plt.axhline(spcz_loc_pi.mean(), color='tab:green', lw=0.5)
plt.axhline(spcz_loc_pi.mean(), color='tab:green', lw=0.5)
plt.title('SPCZ location')
plt.minorticks_on()
plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.xlim(-2, 13)
plt.legend([bp1["boxes"][0], bp2["boxes"][0]], ['Transient', 'NZ'], loc='upper left', frameon=False, fontsize=10)

In [ ]:
nino34_mag_trans = rolling_window_std(funcs.detrend_rolling_window(ts_trans.sel(lat = slice(-5, 5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15), window_size=30)
nino34_mag_stable = rolling_window_std(funcs.detrend_rolling_window(ts_stable.sel(lat = slice(-5, 5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15), window_size=30)

In [ ]:
pac_sstg_trans = rolling_window_smoothing(ts_trans.sel(lat = slice(-5, 5), lon = slice(140, 170)).weighted(weights_model).mean(('lat', 'lon')) - 
                                          ts_trans.sel(lat = slice(-5, 5), lon = slice(190, 270)).weighted(weights_model).mean(('lat', 'lon')), window_size=30)
pac_sstg_stable = rolling_window_smoothing(ts_stable.sel(lat = slice(-5, 5), lon = slice(140, 170)).weighted(weights_model).mean(('lat', 'lon')) - 
                                           ts_stable.sel(lat = slice(-5, 5), lon = slice(190, 270)).weighted(weights_model).mean(('lat', 'lon')), window_size=30)

In [ ]:
test = xr.open_mfdataset('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/piControl/r1i1p1f1/Amon/ts/gn/latest/*.nc')

In [ ]:
import xesmf as xe

In [ ]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

In [ ]:
regridder = xe.Regridder(test, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)

In [ ]:
test_loaded = regridder(funcs.calc_anom(test.ts, test.ts).resample(time = 'AS-JUN').mean('time')).sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).mean(('lat', 'lon')).load()
test_loaded

In [ ]:
test1 = regridder(funcs.calc_anom(test.ts, test.ts).resample(time = 'AS-JUN').mean('time'))
test1_loaded = (test1.sel(lat=slice(-5,5), lon = slice(140, 170)).mean(('lat', 'lon')) - test1.sel(lat=slice(-5,5), lon = slice(190, 270)).mean(('lat', 'lon'))).load()
test1_loaded

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(4,4))
rolling_std_picontrol = rolling_window_std(funcs.detrend_rolling_window(test_loaded, window_size=15), window_size=30)
plt.plot(rolling_std_picontrol.time.dt.year, rolling_std_picontrol, label='piControl')
plt.axhline(rolling_std_picontrol.quantile(0.1, dim='time'))
plt.axhline(rolling_std_picontrol.quantile(0.9, dim='time'))
plt.ylim(0, 1.2)
sns.despine()
plt.minorticks_on()
plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.legend(frameon=False, fontsize=11)
plt.ylabel('ENSO variability (degC)')
plt.xlabel('Time')


In [ ]:
plt.figure(figsize=(7,4))

plt.plot(nino34_mag_stable.time.dt.year, nino34_mag_stable.sel(model = 'B2030'), color='gold', lw=2.0, alpha=0.7, label='B2030')
plt.axhline(nino34_mag_stable.sel(model='B2030').quantile(0.1, dim='time'), color='gold')
plt.axhline(nino34_mag_stable.sel(model='B2030').quantile(0.9, dim='time'), color='gold')
plt.plot(nino34_mag_stable.time.dt.year, nino34_mag_stable.sel(model = 'B2045'), color='tab:red', lw=2.0, alpha=0.7, label='B2045')
plt.axhline(nino34_mag_stable.sel(model='B2045').quantile(0.1, dim='time'), color='tab:red')
plt.axhline(nino34_mag_stable.sel(model='B2045').quantile(0.9, dim='time'), color='tab:red')
plt.plot(nino34_mag_stable.time.dt.year, nino34_mag_stable.sel(model = 'B2060'), color='tab:brown', lw=2.0, alpha=0.7, label='B2060')
plt.axhline(nino34_mag_stable.sel(model='B2060').quantile(0.1, dim='time'), color='tab:brown')
plt.axhline(nino34_mag_stable.sel(model='B2060').quantile(0.9, dim='time'), color='tab:brown')

plt.plot(nino34_mag_trans.time.dt.year, nino34_mag_trans.quantile(0.5, 'model'), color='k', lw=2.0, label='hist+ssp585')
plt.plot(nino34_mag_trans.time.dt.year, nino34_mag_trans.sel(model='ACCESS-ESM1-5_r10i1p1f1'), color='k', ls='--', lw=1.0, label='hist+ssp5 r10')
plt.fill_between(nino34_mag_trans.time.dt.year, nino34_mag_trans.quantile(0.1, 'model'), nino34_mag_trans.quantile(0.9, 'model'), color='k', alpha=0.3)
# plt.axhline(rolling_std_picontrol.quantile(0.1, dim='time'))
# plt.axhline(rolling_std_picontrol.quantile(0.9, dim='time'))
plt.ylim(0, 1.2)
sns.despine(left=True, trim=True)
plt.yticks([])
# plt.minorticks_on()
# plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.legend(frameon=False, fontsize=11, loc='lower right', ncol=2)
# plt.ylabel('ENSO variability (degC)')
plt.xlabel('Year')


In [ ]:
plt.figure(figsize=(4,4))
rolling_sstg_picontrol = rolling_window_smoothing(test1_loaded, window_size=30)
plt.plot(rolling_sstg_picontrol.time.dt.year[30:], rolling_sstg_picontrol[30:], label='piControl')
plt.axhline(rolling_sstg_picontrol[30:].quantile(0.5, dim='time'))
plt.axhline(0.0, color='k', lw=0.5)
plt.ylim(-2, 0.5)
sns.despine()
plt.minorticks_on()
plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.legend(frameon=False, fontsize=11)
plt.ylabel('Pacific SST gradient (degC)')
plt.xlabel('Time')


In [ ]:
plt.figure(figsize=(7,4))

plt.plot(pac_sstg_stable.time.dt.year, pac_sstg_stable.sel(model = 'B2030'), color='gold', lw=2.0, alpha=0.7, label='B2030')
plt.axhline(pac_sstg_stable.sel(model='B2030').quantile(0.5, dim='time'), color='gold')
plt.plot(pac_sstg_stable.time.dt.year, pac_sstg_stable.sel(model = 'B2045'), color='tab:red', lw=2.0, alpha=0.7, label='B2045')
plt.axhline(pac_sstg_stable.sel(model='B2045').quantile(0.5, dim='time'), color='tab:red')
plt.plot(pac_sstg_stable.time.dt.year, pac_sstg_stable.sel(model = 'B2060'), color='tab:brown', lw=2.0, alpha=0.7, label='B2060')
plt.axhline(pac_sstg_stable.sel(model='B2060').quantile(0.5, dim='time'), color='tab:brown')

plt.plot(pac_sstg_trans.time.dt.year, pac_sstg_trans.quantile(0.5, 'model'), color='k', lw=2.0, label='hist+ssp585')
plt.plot(pac_sstg_trans.time.dt.year, pac_sstg_trans.sel(model = 'ACCESS-ESM1-5_r10i1p1f1'), color='k', ls='--', lw=1.0, label='hist+ssp585 r10')
plt.fill_between(pac_sstg_trans.time.dt.year, pac_sstg_trans.quantile(0.1, 'model'), pac_sstg_trans.quantile(0.9, 'model'), color='k', alpha=0.3)
# plt.axhline(rolling_std_picontrol.quantile(0.1, dim='time'))
# plt.axhline(rolling_std_picontrol.quantile(0.9, dim='time'))
plt.ylim(-2, 0.5)
sns.despine(left=True, trim=True)
plt.yticks([])
# plt.minorticks_on()
# plt.gca().tick_params(bottom=False, axis='x', which='minor')
plt.legend(frameon=False, fontsize=11, loc='lower right', ncol=2)
plt.axhline(0.0, color='k', lw=0.5)
# plt.ylabel('ENSO variability (degC)')
plt.xlabel('Year')


In [ ]:
nino34_mag_pi = funcs.detrend_rolling_window(ts_trans.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')).sel(time = slice('1850', '1900')), window_size=15).std('time')
nino34_mag15 = funcs.detrend_rolling_window(ts_trans.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_trans > 1.3) & (gmst_trans < 1.7), drop=True).std('time')
nino34_mag20 = funcs.detrend_rolling_window(ts_trans.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_trans > 1.9) & (gmst_trans < 2.1), drop=True).std('time')
nino34_mag30 = funcs.detrend_rolling_window(ts_trans.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_trans > 2.9) & (gmst_trans < 3.1), drop=True).std('time')

In [ ]:
nino34_mag15_stable = funcs.detrend_rolling_window(ts_stable.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_stable > 1.3) & (gmst_stable < 1.7), drop=True).std('time')
nino34_mag20_stable = funcs.detrend_rolling_window(ts_stable.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_stable > 1.9) & (gmst_stable < 2.1), drop=True).std('time')
nino34_mag30_stable = funcs.detrend_rolling_window(ts_stable.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights_model).mean(('lat', 'lon')), window_size=15).where((gmst_stable > 2.9) & (gmst_stable < 3.1), drop=True).std('time')

In [ ]:
plt.boxplot([nino34_mag30, nino34_mag30_stable], whis=[5,95])

In [ ]:
d15 = (np.abs(itcz_loc_15) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_15) + (spcz_loc_pi))
d20 = (np.abs(itcz_loc_20) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_20) + (spcz_loc_pi))
d30 = (np.abs(itcz_loc_30) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_30) + (spcz_loc_pi))
d40 = (np.abs(itcz_loc_40) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_40) + (spcz_loc_pi))

In [ ]:
# d15_stable = (np.abs(itcz_loc_15_stable) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_15_stable) + (spcz_loc_pi))
# d20_stable = (np.abs(itcz_loc_20_stable) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_20_stable) + (spcz_loc_pi))
# d30_stable = (np.abs(itcz_loc_30_stable) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_30_stable) + (spcz_loc_pi))
# # d40_stable = (np.abs(itcz_loc_40_stable) - np.abs(itcz_loc_pi)) + (np.abs(spcz_loc_40_stable) + (spcz_loc_pi))

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.boxplot([d15, d20, d30, d40], whis=[5,95], widths=0.2, boxprops=dict(color='k', lw=2.0), medianprops=dict(color='k'), meanprops=dict(marker='^'), showmeans=True, flierprops=dict(marker='+'))
plt.gca().set_xticklabels(['GWL1.5', 'GWL2.0', 'GWL3.0', 'GWL4.0'])
plt.ylabel('Latitude ($^{o}$N)')
plt.title('Equatorward shift of convection zones')
plt.axhline(0.0, color='k', lw=0.5)
plt.minorticks_on()
plt.gca().tick_params(bottom=False, axis='x', which='minor')

In [ ]:
hist_files = []
ssp_files = []
for i in range(1, 41):
    hist_files.append(xr.open_mfdataset(f'/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r{i}i1p1f1/Amon/vas/gn/latest/*.nc', use_cftime=True))
    ssp_files.append(xr.open_mfdataset(f'/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r{i}i1p1f1/Amon/vas/gn/latest/*.nc', use_cftime=True))
    vas_hist = xr.concat(hist_files, "model")
    vas_ssp = xr.concat(ssp_files, "model")
    # uas_hist['model'] = np.arange(1,41)
    # uas_ssp['model'] = np.arange(1,41)
    # uas = xr.concat([uas_hist, uas_ssp], dim='time')

In [ ]:
vas = xr.concat([vas_hist, vas_ssp], 'time')
vas['model'] = np.arange(1, 41, 1)

In [ ]:
vas = xr.concat([vas_hist, vas_ssp], 'time')
vas['model'] = np.arange(1, 41, 1)

In [ ]:
##  next steps are to regrid and then resample and then save the files 